[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/04_tool_packaging/04_tool_packaging.ipynb)

# 04 · 工具打包与插件（plugins / manifest / 命名空间 / 版本）

目标：用**纯标准库**（只用 `json`，semver 比较器**亲手写**、不依赖 `packaging`）从零造一个插件系统——**manifest 解析与校验 → 插件发现 → 插件加载(把能力接入注册表) → 命名空间隔离 → semver 版本约束 → 依赖解析(拓扑序+版本满足)**，全程 `assert` 验证、**无需 API key、不依赖框架**。

路线：manifest 解析/校验 → 三大注册表 → 插件加载器 → 命名空间隔离 → semver 解析+比较 → 版本约束匹配 → 依赖解析 → ✏️ 练习 → 📖 答案 → 🧪 真实 plugin.json 胶囊。

> 心智模型：**插件 = 一份 manifest（声明提供什么、依赖什么）+ 一套加载逻辑（把能力接入注册表、避免冲突）**。难点在命名空间隔离（多盒共存）与版本依赖（盒与盒兼容）。

## 1 · manifest 解析与校验

一个插件的一切从 manifest 开始（Claude Code 用 `.claude-plugin/plugin.json`）。加载器**先读 manifest、再决定怎么装**。

manifest 声明：`name`(唯一标识/命名空间前缀)、`version`(semver)、`description`、以及提供的 `skills`/`commands`/`mcpServers` 与 `dependencies`。

解析 = JSON 反序列化 + **最小校验**：必填字段在不在？version 合法吗？校验不过要**明确报错指出哪个字段**，而非把残缺 manifest 装进系统。

In [ ]:
import json, re

_SEMVER_RE = re.compile(r'^\d+\.\d+\.\d+$')   # MAJOR.MINOR.PATCH (本课核心子集)

def parse_manifest(text):
    '''解析 manifest(JSON 文本) -> dict, 并做最小校验。校验失败抛 ValueError(指出字段)。'''
    try:
        m = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f'manifest 不是合法 JSON: {e}')
    if not isinstance(m, dict):
        raise ValueError('manifest 顶层必须是对象')
    # 必填字段
    for field in ('name', 'version'):
        if field not in m:
            raise ValueError(f'manifest 缺少必填字段: {field}')
    if not _SEMVER_RE.match(str(m['version'])):
        raise ValueError(f"version 不是合法 semver(MAJOR.MINOR.PATCH): {m['version']!r}")
    # 可选能力字段给默认值, 统一形状(便于后续加载)
    m.setdefault('description', '')
    m.setdefault('skills', [])
    m.setdefault('commands', [])
    m.setdefault('mcpServers', {})
    m.setdefault('dependencies', {})
    return m

good = '{"name": "py-helper", "version": "1.3.0", "skills": ["docstring", "tests"], "commands": ["lint"]}'
m = parse_manifest(good)
print('解析成功:', m['name'], m['version'], '| skills:', m['skills'], '| commands:', m['commands'])
assert m['name'] == 'py-helper' and m['version'] == '1.3.0'
assert m['skills'] == ['docstring', 'tests'] and m['dependencies'] == {}   # 默认补全

# 校验应当拦住残缺/非法 manifest, 并指出字段
for bad, why in [('{"version": "1.0.0"}', 'name'),
                 ('{"name": "x"}', 'version'),
                 ('{"name": "x", "version": "1.0"}', 'semver'),
                 ('not json', 'JSON')]:
    try:
        parse_manifest(bad); raise AssertionError('应当报错: ' + why)
    except ValueError as e:
        print('  正确拦截:', str(e)[:48])
print('✅ manifest 解析+校验: 合法解析、残缺/非法精确报错')

## 2 · 三大注册表：插件能力的归宿

插件加载，本质是**把 manifest 声明的能力逐一接入对应注册表**。本课的 agent 有三大注册表（呼应模块 01/02/03）：

- **skill 注册表**：`name → skill`（模块 01）
- **命令路由**：`name → command`（模块 02）
- **工具/MCP 注册表**：`name → 工具或 server 配置`（模块 03）

先把这三张表写成一个极简容器——后面的加载器就是往它们里 `register`。

In [ ]:
class Registries:
    '''agent 的三大注册表。插件加载器把能力接入这里。'''
    def __init__(self):
        self.skills = {}      # name -> skill 内容
        self.commands = {}    # name -> 命令模板
        self.mcp = {}         # name -> MCP server 配置
    def summary(self):
        return {'skills': sorted(self.skills), 'commands': sorted(self.commands),
                'mcp': sorted(self.mcp)}

reg = Registries()
reg.skills['demo'] = '一个示例 skill 正文'
print('三大注册表初始化:', reg.summary())
assert reg.summary() == {'skills': ['demo'], 'commands': [], 'mcp': []}
print('✅ 三大注册表就位：skills / commands / mcp，加载器将往这里接入能力')

## 3 · 命名空间隔离：让插件互不打架

多插件共存的头号障碍：两个插件都提供 `format` 命令怎么办？不处理则后者**静默覆盖**前者——危险！

解法：**给每个能力加 `插件名:` 前缀**。`py-helper:format` 与 `git-pro:format` 井水不犯河水。

区分两层冲突：**跨插件同名**(加前缀后不再冲突, 共存) vs **同一插件内同名**(真错误, 报错拒绝)。

In [ ]:
def register_namespaced(registry, plugin_name, cap_name, value):
    '''带命名空间地注册一个能力。key = 插件名:能力名。
       同一(插件,能力)重复注册 -> 真冲突, 报错(前缀也救不了)。'''
    key = f'{plugin_name}:{cap_name}'
    if key in registry:
        raise ValueError(f'命名冲突: {key} 已存在')
    registry[key] = value
    return key

ns = {}
# 跨插件同名 format -> 加前缀后共存, 不冲突
k1 = register_namespaced(ns, 'py-helper', 'format', '<py-helper 的 format>')
k2 = register_namespaced(ns, 'git-pro', 'format', '<git-pro 的 format>')
print('两个 format 共存:', sorted(ns))
assert k1 == 'py-helper:format' and k2 == 'git-pro:format'
assert ns['py-helper:format'] != ns['git-pro:format']   # 没有互相覆盖!

# 同一插件内重名 -> 真冲突, 必须报错
raised = False
try:
    register_namespaced(ns, 'py-helper', 'format', '<重复!>')
except ValueError as e:
    raised = True; print('正确拒绝同插件内重名:', e)
assert raised, '同一插件内重名应当报错, 而非静默覆盖'
print('✅ 命名空间隔离: 跨插件同名无痛共存、同插件内重名明确失败')

## 4 · 插件加载器：把 manifest 翻译成一串 register

现在把前三节拼起来：**插件加载器 = 一个把 manifest 翻译成「对各注册表的一串带命名空间的 register 调用」的翻译器**。

加载一个插件 = 读它 manifest 的 `skills`/`commands`/`mcpServers`，逐一带前缀注册进三大注册表。

In [ ]:
def load_plugin(manifest, registries):
    '''按 manifest 把插件的能力接入三大注册表(全部带命名空间前缀)。
       返回接入的能力 key 列表。冲突会由 register_namespaced 抛出。'''
    name = manifest['name']
    loaded = []
    for s in manifest.get('skills', []):
        loaded.append(register_namespaced(registries.skills, name, s, f'<{name} 的 skill: {s}>'))
    for cmd in manifest.get('commands', []):
        loaded.append(register_namespaced(registries.commands, name, cmd, f'<{name} 的命令: {cmd}>'))
    for srv_name, srv_cfg in manifest.get('mcpServers', {}).items():
        loaded.append(register_namespaced(registries.mcp, name, srv_name, srv_cfg))
    return loaded

reg2 = Registries()
py = parse_manifest('{"name":"py-helper","version":"1.3.0","skills":["docstring","tests"],"commands":["lint"],"mcpServers":{"fs":{"cmd":"fs-server"}}}')
git = parse_manifest('{"name":"git-pro","version":"2.0.0","commands":["lint","status"]}')   # 注意 git-pro 也有 lint!
loaded_py = load_plugin(py, reg2)
loaded_git = load_plugin(git, reg2)
print('py-helper 接入:', loaded_py)
print('git-pro  接入:', loaded_git)
print('合并后命令路由:', sorted(reg2.commands))
# 两个插件各自的 lint 都在, 靠前缀区分, 没有互相覆盖
assert 'py-helper:lint' in reg2.commands and 'git-pro:lint' in reg2.commands
assert reg2.skills == {'py-helper:docstring': '<py-helper 的 skill: docstring>',
                       'py-helper:tests': '<py-helper 的 skill: tests>'}
assert reg2.mcp == {'py-helper:fs': {'cmd': 'fs-server'}}
print('✅ 插件加载器: 把 manifest 声明的能力带命名空间接入三大注册表, 两插件 lint 共存')

## 5 · semver：解析、比较、版本约束匹配

插件依赖要说清**版本**：`core 1.0` 和 `core 2.0` 接口可能完全不同。表达版本用 **semver**：`MAJOR.MINOR.PATCH`。

依赖方用**约束**表达能接受哪些版本：`1.2.3`(正好) / `>=1.2.0`(不低于) / `^1.2.3`(兼容 1.x) / `~1.2.3`(兼容 1.2.x)。

比较器要做两件事：**解析**(`'1.2.3'`→`(1,2,3)`) 与 **匹配**(某版本满足某约束吗)。`^`/`~` 转成「下界+上界」区间。**亲手写, 不用 `packaging`**。

In [ ]:
def parse_version(v):
    '''解析 semver 字符串 -> (major, minor, patch) 三元组(可字典序比较)。'''
    parts = str(v).split('.')
    if len(parts) != 3 or not all(x.isdigit() for x in parts):
        raise ValueError(f'非法 semver: {v!r}')
    return tuple(int(x) for x in parts)

def satisfies(version, constraint):
    '''判断 version 是否满足 constraint。支持: 精确 / >= / ^ / ~。
       三元组天然可字典序比较: (1,2,3) < (1,3,0) < (2,0,0)。'''
    v = parse_version(version)
    constraint = constraint.strip()
    if constraint.startswith('>='):
        return v >= parse_version(constraint[2:].strip())
    if constraint.startswith('^'):          # ^1.2.3: >=1.2.3 且 <2.0.0 (MAJOR 不变)
        lo = parse_version(constraint[1:])
        return lo <= v < (lo[0] + 1, 0, 0)
    if constraint.startswith('~'):          # ~1.2.3: >=1.2.3 且 <1.3.0 (MINOR 不变)
        lo = parse_version(constraint[1:])
        return lo <= v < (lo[0], lo[1] + 1, 0)
    return v == parse_version(constraint)   # 精确

assert parse_version('1.2.3') == (1, 2, 3)
assert parse_version('1.2.3') < parse_version('1.10.0')   # 数值比较, 不是字符串!
# 精确
assert satisfies('1.2.3', '1.2.3') and not satisfies('1.2.4', '1.2.3')
# >=
assert satisfies('2.0.0', '>=1.2.0') and not satisfies('1.1.0', '>=1.2.0')
# ^1.2.3: 收 1.x>=1.2.3, 拒 2.0.0
assert satisfies('1.9.9', '^1.2.3') and not satisfies('2.0.0', '^1.2.3')
assert not satisfies('1.2.2', '^1.2.3')   # 低于下界
# ~1.2.3: 只收 1.2.x, 拒 1.3.0
assert satisfies('1.2.9', '~1.2.3') and not satisfies('1.3.0', '~1.2.3')
print('解析:', parse_version('1.10.0'), '| 1.9.9 满足 ^1.2.3:', satisfies('1.9.9', '^1.2.3'))
print('✅ semver: 解析为三元组、数值比较、精确/>=/^/~ 约束匹配全对')

## 6 · 依赖解析：加载顺序 + 版本满足

插件间有依赖（`py-test` 依赖 `core >=1.0.0`）。依赖解析要做：
1. **加载顺序**：被依赖的先加载 —— 这是**拓扑排序**（依赖图的有向无环排序）。
2. **版本满足**：每个依赖的版本约束都被满足吗？用第 5 节的 `satisfies` 逐条套。
3. 发现 **缺失依赖** 与 **版本冲突**，明确报错。

In [ ]:
def resolve_load_order(manifests):
    '''manifests: {name: manifest}。返回满足依赖的加载顺序(被依赖者在前)。
       同时检查依赖存在且版本满足。缺失/冲突/环 -> ValueError。'''
    # 先做版本与存在性检查
    for name, m in manifests.items():
        for dep, constraint in m.get('dependencies', {}).items():
            if dep not in manifests:
                raise ValueError(f'{name} 依赖缺失: {dep}')
            if not satisfies(manifests[dep]['version'], constraint):
                raise ValueError(f'{name} 依赖版本不满足: {dep} 需要 {constraint}, '
                                 f"实际 {manifests[dep]['version']}")
    # 拓扑排序(DFS), 顺带检测环
    order, state = [], {}   # state: 0=未访问 1=访问中 2=完成
    def visit(n):
        if state.get(n) == 2:
            return
        if state.get(n) == 1:
            raise ValueError(f'依赖成环, 涉及: {n}')
        state[n] = 1
        for dep in manifests[n].get('dependencies', {}):
            visit(dep)            # 先访问被依赖者
        state[n] = 2
        order.append(n)           # 完成后加入 -> 被依赖者先入列
    for n in manifests:
        visit(n)
    return order

core = parse_manifest('{"name":"core","version":"1.2.0"}')
test = parse_manifest('{"name":"py-test","version":"1.0.0","dependencies":{"core":">=1.0.0"}}')
lint = parse_manifest('{"name":"py-lint","version":"1.0.0","dependencies":{"core":"^1.2.0","py-test":">=1.0.0"}}')
mans = {'core': core, 'py-test': test, 'py-lint': lint}
order = resolve_load_order(mans)
print('加载顺序:', order)
# core 必须在 py-test 前, py-test 必须在 py-lint 前
assert order.index('core') < order.index('py-test') < order.index('py-lint')

# 版本冲突应被发现
bad = {'core': parse_manifest('{"name":"core","version":"0.9.0"}'),
       'app': parse_manifest('{"name":"app","version":"1.0.0","dependencies":{"core":">=1.0.0"}}')}
conflict = False
try:
    resolve_load_order(bad)
except ValueError as e:
    conflict = True; print('正确发现版本冲突:', e)
assert conflict
print('✅ 依赖解析: 拓扑序(被依赖者先)、版本满足检查、缺失/冲突/环明确报错')

---
## ✏️ 练习 1：manifest 校验——能力字段类型

第 1 节的 `parse_manifest` 没校验能力字段的**类型**。一个 manifest 若把 `skills` 写成字符串（而非数组）、或 `mcpServers` 写成数组（而非对象），现在会蒙混过关。

实现 `validate_types(m)`：在 `parse_manifest` 之后调用，额外检查 `skills`/`commands` 必须是 `list`、`mcpServers`/`dependencies` 必须是 `dict`；不对则返回 `(False, '字段 X 应为 列表/对象')`，全对返回 `(True, None)`。

In [ ]:
def validate_types(m):
    # m 是 parse_manifest 的输出(已补全默认值)
    # TODO: 检查 skills/commands 是 list, mcpServers/dependencies 是 dict
    #   第一个不对的返回 (False, f'字段 {k} 应为 列表') 或 '... 应为 对象'
    #   全对返回 (True, None)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ok_m = parse_manifest('{"name":"x","version":"1.0.0","skills":["a"],"mcpServers":{"s":{}}}')
assert validate_types(ok_m) == (True, None)
bad1 = parse_manifest('{"name":"x","version":"1.0.0"}'); bad1['skills'] = 'oops'  # 应为 list
ok, err = validate_types(bad1)
assert ok is False and 'skills' in err and '列表' in err
bad2 = parse_manifest('{"name":"x","version":"1.0.0"}'); bad2['mcpServers'] = ['oops']  # 应为 dict
ok2, err2 = validate_types(bad2)
assert ok2 is False and 'mcpServers' in err2 and '对象' in err2
print('✅ 练习 1 通过：能力字段类型校验(list/dict)且精确指出字段')

## ✏️ 练习 2：禁用某个插件——按命名空间卸载

插件要能**启停**：禁用一个插件 = 把它接入的所有能力从注册表里移除。靠命名空间前缀很好做——所有 `插件名:` 开头的 key 都属于它。

实现 `unload_plugin(plugin_name, registries)`：从三大注册表里删除所有以 `plugin_name + ':'` 开头的 key，返回被删除的 key 总数。

In [ ]:
def unload_plugin(plugin_name, registries):
    # TODO: 从 registries.skills/commands/mcp 里删除所有以 f'{plugin_name}:' 开头的 key
    #   返回删除的 key 总数(三张表合计)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
reg3 = Registries()
load_plugin(parse_manifest('{"name":"py-helper","version":"1.0.0","skills":["a","b"],"commands":["lint"]}'), reg3)
load_plugin(parse_manifest('{"name":"git-pro","version":"1.0.0","commands":["status"]}'), reg3)
n = unload_plugin('py-helper', reg3)
assert n == 3, f'py-helper 有 2 skill + 1 command = 3 个能力, 实删 {n}'
# py-helper 的都没了, git-pro 的还在
assert not any(k.startswith('py-helper:') for k in reg3.skills)
assert not any(k.startswith('py-helper:') for k in reg3.commands)
assert 'git-pro:status' in reg3.commands
print('✅ 练习 2 通过：按命名空间前缀卸载插件, 只移除它自己的能力')

## ✏️ 练习 3：semver 约束——补上 `<` 与 `<=`

第 5 节的 `satisfies` 支持 `精确 / >= / ^ / ~`，但少了「上界」运算符。补上 `<`(严格小于) 与 `<=`(不大于)。

实现 `satisfies2(version, constraint)`：在原有基础上额外支持 `<1.2.0`、`<=1.2.0`。
（提示：注意先判更长的前缀 `<=` 再判 `<`，否则 `<=` 会被 `<` 抢先匹配。`>=` 同理已在原函数里。）

In [ ]:
def satisfies2(version, constraint):
    # TODO: 复用 parse_version; 先处理 '<=' 再 '<', 然后回退到原有 satisfies 的逻辑
    #   (>= / ^ / ~ / 精确)。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert satisfies2('1.1.0', '<1.2.0') and not satisfies2('1.2.0', '<1.2.0')
assert satisfies2('1.2.0', '<=1.2.0') and not satisfies2('1.2.1', '<=1.2.0')
# 原有约束仍然工作
assert satisfies2('1.9.9', '^1.2.3') and not satisfies2('2.0.0', '^1.2.3')
assert satisfies2('2.0.0', '>=1.2.0') and satisfies2('1.2.3', '1.2.3')
print('✅ 练习 3 通过：补上 < / <= 上界, 且不破坏原有 >= / ^ / ~ / 精确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def validate_types(m):
    for k in ('skills', 'commands'):
        if not isinstance(m.get(k, []), list):
            return False, f'字段 {k} 应为 列表'
    for k in ('mcpServers', 'dependencies'):
        if not isinstance(m.get(k, {}), dict):
            return False, f'字段 {k} 应为 对象'
    return True, None

In [ ]:
# 练习 2 参考答案
def unload_plugin(plugin_name, registries):
    prefix = f'{plugin_name}:'
    total = 0
    for registry in (registries.skills, registries.commands, registries.mcp):
        doomed = [k for k in registry if k.startswith(prefix)]
        for k in doomed:
            del registry[k]
        total += len(doomed)
    return total

In [ ]:
# 练习 3 参考答案
def satisfies2(version, constraint):
    v = parse_version(version)
    constraint = constraint.strip()
    if constraint.startswith('<='):
        return v <= parse_version(constraint[2:].strip())
    if constraint.startswith('<'):
        return v < parse_version(constraint[1:].strip())
    return satisfies(version, constraint)   # 回退到原有 >= / ^ / ~ / 精确

---
## 🧪 真实数据胶囊：真实形状的 plugin.json

下面是一个**贴近真实**的 Claude Code 插件 manifest（形如 `.claude-plugin/plugin.json`）。我们用上面亲手写的 `parse_manifest` 解析它、用 `load_plugin` 加载它，体会从零实现与真实生态的对应。

> 形状对照：真实 Claude Code 插件用 `name`/`version`/`description` 做身份，用 `skills`/`commands`/`mcpServers` 声明能力——和本课一致。

In [ ]:
# 真实形状的 plugin.json (贴近 Claude Code .claude-plugin/plugin.json)
REAL_MANIFEST = '''{
  "name": "python-toolkit",
  "version": "2.1.0",
  "description": "Python project assistant: docstrings, tests, linting",
  "skills": ["write-docstring", "organize-tests", "commit-style"],
  "commands": ["test", "lint", "format"],
  "mcpServers": {
    "filesystem": {"command": "mcp-server-filesystem", "args": ["--root", "."]}
  },
  "dependencies": {"core-utils": ">=1.0.0"}
}'''

m = parse_manifest(REAL_MANIFEST)
print('插件:', m['name'], 'v' + m['version'])
print('描述:', m['description'])
print('提供 skills:', m['skills'])
print('提供 commands:', m['commands'])
print('携带 MCP server:', list(m['mcpServers']))
print('依赖:', m['dependencies'])
# 用本课加载器把它接入(注: 它依赖 core-utils, 单独加载时不检查依赖, 仅接入能力)
real_reg = Registries()
loaded = load_plugin(m, real_reg)
assert 'python-toolkit:test' in real_reg.commands
assert 'python-toolkit:write-docstring' in real_reg.skills
assert 'python-toolkit:filesystem' in real_reg.mcp
assert len(loaded) == 3 + 3 + 1   # 3 skills + 3 commands + 1 mcp server
print('✅ 本课解析器/加载器直接适用于真实形状 plugin.json, 共接入', len(loaded), '个能力')

**🧪 胶囊练习**：实现 `count_capabilities(manifest)`：给定一个 manifest，返回它声明的能力**总数**（`skills` 数 + `commands` 数 + `mcpServers` 数）。（真实 marketplace 统计「这个插件提供多少能力」就是这么做的。）

In [ ]:
def count_capabilities(manifest):
    # TODO: 返回 len(skills) + len(commands) + len(mcpServers)
    raise NotImplementedError

In [ ]:
# 自测
cnt = count_capabilities(parse_manifest(REAL_MANIFEST))
assert cnt == 7, f'3 skills + 3 commands + 1 mcp = 7, got {cnt}'
print('python-toolkit 提供能力总数:', cnt)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def count_capabilities(manifest):
    return (len(manifest.get('skills', [])) + len(manifest.get('commands', []))
            + len(manifest.get('mcpServers', {})))

---
## 🔧 旁注：加载好的插件，怎么接到真实 Claude

插件加载后，三大注册表就备好了能力。接真实 Claude 时，**工具注册表里的工具变成 `tools=[...]`，命名空间前缀映射到工具名**（伪代码，**本环境不跑、需 API key**）：

```python
import os, anthropic

def to_anthropic_tools(registries):
    '''把插件加载后的工具注册表转成 Anthropic tools 参数。
       命名空间 key 'py-helper:format' -> 工具名(冒号可保留或转下划线)。'''
    tools = []
    for key, cfg in registries.mcp.items():        # 这里以 mcp/工具为例
        tools.append({'name': key.replace(':', '__'),
                      'description': cfg.get('description', key),
                      'input_schema': cfg.get('input_schema', {'type': 'object', 'properties': {}})})
    return tools

if os.environ.get('ANTHROPIC_API_KEY'):
    client = anthropic.Anthropic()                 # 读 ANTHROPIC_API_KEY
    resp = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=1024,
        tools=to_anthropic_tools(registries),      # 插件提供的工具, 原样可用!
        messages=[{'role': 'user', 'content': '...'}],
    )
else:
    pass   # 无 key -> 用本课 MockLLM, 绝不阻断 (见模块 00 的 make_llm)
```

对应关系：插件 manifest 声明的工具 ↔ `tools=[...]`、命名空间前缀 ↔ 工具名（避免不同插件工具撞名）、本课加载/隔离逻辑**原样适用**。真实 Claude Code 还会把插件携带的 `mcpServers` 直接接入（见模块 03 的 MCP），把 `commands` 注册成 `/插件名:命令`。这就是「插件系统可迁移」的含义。

### 小结
- 插件 = **manifest(声明提供什么/依赖什么) + 加载逻辑(把能力接入注册表、避免冲突)**，是 skill/命令/MCP 工具的**分发层**。
- **manifest 先解析+校验**：必填(name/version)、version 合法 semver、能力字段类型；残缺要精确报错。
- **发现 vs 加载**：发现只读 manifest 建目录(轻); 加载把选中插件的能力接入三大注册表(重、按需) —— 又是渐进披露。
- **命名空间隔离是安全底线**：跨插件同名加前缀共存、同插件内重名报错; 绝不静默覆盖(否则能力被掉包)。
- **semver**：MAJOR.MINOR.PATCH; 约束 `>=`/`^`/`~`/精确; 解析为三元组数值比较; 亲手写不靠 `packaging`。
- **依赖解析**：拓扑序(被依赖者先) + 逐条版本满足; 缺失/冲突/环明确报错。

下一站：**模块 05 · 完整 Skills 系统** —— 把 skill 加载(01)、slash 命令(02)、MCP(03)、插件(04)组装成一个不依赖框架的完整系统，接到 agent 循环，端到端完成一个真实任务。